In [1]:
from ioMicro import *

In [2]:
data_folder = r'Z:\Zane_20CRE\7_17_2024__T7_20CRE'

In [10]:
# map all the hybes
hybes =  glob.glob(data_folder+os.sep+'D*')
# map all the fovs
fovs = [os.path.basename(fl)for fl in glob.glob(hybes[0]+os.sep+'*.zarr')]
def get_Hi(fld): 
    try: return int(os.path.basename(fld)[1:]) 
    except: return -1
    
hybes = np.array(hybes)[np.argsort([get_Hi(hybe) for hybe in hybes ])]

In [11]:
hybes

array(['Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D1',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D2',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D3',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D4',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D5',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D6',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D7'], dtype='<U36')

In [5]:
from scipy.signal import fftconvolve

def fit(hybe,fov,analysis_folder='/projects/ps-renlab2/zgibbs/STARR-FISH/10_12_2023__CRE-20_RNASEQ',icol=0,redo=False,hth = 50,plt_val=True):
    fl = hybe+os.sep+fov
    tag = os.path.basename(hybe)
    save_file=analysis_folder+os.sep+fov.split('.')[0]+'--'+tag+'_fits_icol'+str(icol)+'.npz'
    if not os.path.exists(save_file) or redo:
        im = read_im(fl)
        ncols,sz,sx,sy = im.shape
        #for icol in np.arange(ncols-1):
        imS = np.array(im[icol],dtype=np.float32) #GFP cy5 half
        #Convolve with gaussian 
        sz=5
        X = np.indices([2*sz+1]*3)-sz
        sigma=1.5
        gaussian = np.exp(-np.sum(X**2,axis=0)/2/sigma**2)
        im_conv = fftconvolve(imS,gaussian,mode='same')/np.sum(gaussian)
        #Subtract local background
        im_conv_ = norm_slice(im_conv,s=30)
        #fit the spots
        Xh = get_local_max(im_conv_,hth,im_raw=im_conv,delta=1,delta_fit=3)
        if plt_val:
            import napari
            v = napari.view_image(im_conv_)
            v.add_image(imS)
            h = Xh[:,-1]
            size = 5+np.clip(h/np.percentile(h,99.9),0,1)*10
            v.add_points(Xh[:,:3],size=size,face_color=[0,0,0,0],edge_color='y')
        np.savez_compressed(save_file,Xh=Xh)

In [6]:
def tag_to_hybe(tag):
    try:
        return int(tag[1:])
    except:
        return -1

def get_XH(save_folder='something', fov='something', ncols=3, chromatic_fl='something', drift_fl='something'):
        """Load in the fitted dots for each field of view.
        Apply chromatic abberation correction and drift correction and append corrected fitted data 
        int self.XH structure"""
        ncols = ncols
        drift_dic = pickle.load(open(drift_fl,'rb'))
        tags = list(drift_dic.keys())
        if not os.path.exists(chromatic_fl):
            self.m=None
        else:
            m = np.load(chromatic_fl)
        XH = []
        save_folder = save_folder
        fov = fov
        for tag in tqdm(tags):
            iH = tag_to_hybe(tag)
            for icol in range(ncols):
                #tag = os.path.basename(fld)#Conv_zscan__30--H_GFP_fits_icol0
                save_fl = save_folder+os.sep+fov.split('.')[0]+'--'+tag+'_fits_icol'+str(icol)+'.npz'
                Xh = np.load(save_fl)['Xh']
                tzxy = drift_dic[tag][0]
                m=None
                if icol==0: m=m
                Xh[:,:3] = apply_colorcor(Xh[:,:3],m=m)
                Xh[:,:3]+=tzxy# drift correction
                #chromatic abberation -to check
                ### took out code for assigning bits to array ###
        XH = np.array(Xh)

In [28]:
def main_do_compute_fits(save_folder,hybe,fov,icol,save_fl,psf,old_method):
    im_ = read_im(hybe+os.sep+fov)
    im__ = np.array(im_[icol],dtype=np.float32)
    
    if old_method:
        ### previous method
        im_n = norm_slice(im__,s=30)
        #Xh = get_local_max(im_n,500,im_raw=im__,dic_psf=None,delta=1,delta_fit=3,dbscan=True,
        #      return_centers=False,mins=None,sigmaZ=1,sigmaXY=1.5)
        Xh = get_local_maxfast_tensor(im_n,th_fit=500,im_raw=im__,dic_psf=None,delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5,gpu=True)
    else:
        ### new method
        fl_med = flat_field_tag+'med_col_raw'+str(icol)+'.npz'
        if os.path.exists(fl_med):
            im_med = np.array(np.load(fl_med)['im'],dtype=np.float32)
            im_med = cv2.blur(im_med,(20,20))
            im__ = im__/im_med*np.median(im_med)
        else:
            print("Did not find flat field")
        try:
            Xh = get_local_max_tile(im__,th=3600,s_ = 500,pad=100,psf=psf,plt_val=None,snorm=30,gpu=True,
                                    deconv={'method':'wiener','beta':0.0001},
                                    delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5)
        except:
            Xh = get_local_max_tile(im__,th=3600,s_ = 500,pad=100,psf=psf,plt_val=None,snorm=30,gpu=False,
                                    deconv={'method':'wiener','beta':0.0001},
                                    delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5)
    np.savez_compressed(save_fl,Xh=Xh)

In [34]:
def get_local_maxfast_tensor(m,th_fit=500,im_raw=None,dic_psf=None,delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5,gpu=False):
    import torch
    dev = "cuda:0" if (torch.cuda.is_available() and gpu) else "cpu"
    im_dif = torch.from_numpy(im_dif_npy).to(dev)
    z,x,y = torch.where(im_dif>th_fit)
    zmax,xmax,ymax = im_dif.shape
    def get_ind(x,xmax):
        # modify x_ to be within image
        x_ = torch.clone(x)
        bad = x_>=xmax
        x_[bad]=xmax-x_[bad]-2
        bad = x_<0
        x_[bad]=-x_[bad]
        return x_
    #def get_ind(x,xmax):return x%xmax
    for d1 in range(-delta,delta+1):
        for d2 in range(-delta,delta+1):
            for d3 in range(-delta,delta+1):
                if (d1*d1+d2*d2+d3*d3)<=(delta*delta):
                    z_ = get_ind(z+d1,zmax)
                    x_ = get_ind(x+d2,xmax)
                    y_ = get_ind(y+d3,ymax)
                    keep = im_dif[z,x,y]>=im_dif[z_,x_,y_]
                    z,x,y = z[keep],x[keep],y[keep]
    h = im_dif[z,x,y]
    
    
    if len(x)==0:
        return []
    if delta_fit>0:
        d1,d2,d3 = np.indices([2*delta_fit+1]*3).reshape([3,-1])-delta_fit
        kp = (d1*d1+d2*d2+d3*d3)<=(delta_fit*delta_fit)
        d1,d2,d3 = d1[kp],d2[kp],d3[kp]
        d1 = torch.from_numpy(d1).to(dev)
        d2 = torch.from_numpy(d2).to(dev)
        d3 = torch.from_numpy(d3).to(dev)
        im_centers0 = (z.reshape(-1, 1)+d1).T
        im_centers1 = (x.reshape(-1, 1)+d2).T
        im_centers2 = (y.reshape(-1, 1)+d3).T
        z_ = get_ind(im_centers0,zmax)
        x_ = get_ind(im_centers1,xmax)
        y_ = get_ind(im_centers2,ymax)
        im_centers3 = im_dif[z_,x_,y_]
        if im_raw is not None:
            im_raw_ = torch.from_numpy(im_raw).to(dev)
            im_centers4 = im_raw_[z_,x_,y_]
            habs = im_raw_[z,x,y]
        else:
            im_centers4 = im_dif[z_,x_,y_]
            habs = x*0
            a = x*0
        Xft = torch.stack([d1,d2,d3]).T
    
        bk = torch.min(im_centers3,0).values
        im_centers3 = im_centers3-bk
        im_centers3 = im_centers3/torch.sum(im_centers3,0)
        if dic_psf is None:
            sigma = torch.tensor([sigmaZ,sigmaXY,sigmaXY],dtype=torch.float32,device=dev)#np.array([sigmaZ,sigmaXY,sigmaXY],dtype=np.flaot32)[np.newaxis]
            Xft_ = Xft/sigma
            norm_G = torch.exp(-torch.sum(Xft_*Xft_,-1)/2.)
            norm_G=(norm_G-torch.mean(norm_G))/torch.std(norm_G)
    
            hn = torch.mean(((im_centers3-im_centers3.mean(0))/im_centers3.std(0))*norm_G.reshape(-1,1),0)
            a = torch.mean(((im_centers4-im_centers4.mean(0))/im_centers4.std(0))*norm_G.reshape(-1,1),0)
            
        zc = torch.sum(im_centers0*im_centers3,0)
        xc = torch.sum(im_centers1*im_centers3,0)
        yc = torch.sum(im_centers2*im_centers3,0)
        Xh = torch.stack([zc,xc,yc,bk,a,habs,hn,h]).T.cpu().detach().numpy()
    else:
        Xh =  torch.stack([z,x,y,h]).T.cpu().detach().numpy()
    return Xh

In [12]:
hybes

array(['Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D1',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D2',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D3',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D4',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D5',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D6',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\D7'], dtype='<U36')

In [13]:
for fov in tqdm(fovs):
    for hybe in hybes:
        for icol in range(3):
            fit(hybe, fov,icol=icol,analysis_folder=r'Z:\Zane_20CRE\7_17_2024__T7_20CRE\analysis',redo=False,hth = 50,plt_val=False)

100%|███████████████████████████████████████████████████████████████████████████| 225/225 [69:16:40<00:00, 1108.45s/it]


In [26]:
fld+os.sep+tag+os.sep+fov

'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H0\\Conv_zscan__000.zarr'

In [27]:
hybe

'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H0'

In [33]:
?get_local_maxfast_tensor

Object `get_local_maxfast_tensor` not found.


In [32]:
import ioMicro